# Learned Quantization & Continuous Ablations: The Bottleneck

## Purpose

Test if:
1. **Continuous MSE heads eliminate manifold fracture entirely** — the
   attention mechanism is innocent; the categorical bottleneck is the
   sole cause.
2. **Learned quantization (VQ-VAE / k-means) merely delays the tax** —
   better tokens improve encoding efficiency but cannot escape the
   logarithmic capacity bound dictated by Shannon's theorem.

## Design

| Condition | Input | Output Head | Loss | Tokens |
|---|---|---|---|---|
| **A: Continuous MSE** | Discrete 256-bin | Linear → scalar | MSE | Uniform grid |
| **B: VQ Learned** | VQ k-means tokens | Categorical logits | CE | Learned codebook |
| **C: Baseline** | Discrete 256-bin | Categorical logits | CE | Uniform grid |

All trained on the **Lorenz attractor** (single oscillator, d_M ≈ 2.06).

## Key Predictions

- **Condition A** (MSE): Smooth orbits, no fracture → proves attention is innocent
- **Condition B** (VQ): Better than C at low params, but STILL fractures → proves
  discrete channel capacity is the bottleneck, not tokenizer quality
- **Condition C** (Baseline): Standard fracture pattern (reference)

---

## Setup

## Install Dependencies

In [ ]:
print('Installing core dependencies...')
!pip install -q torch matplotlib seaborn pandas scipy numpy scikit-learn

print('\nBuilding mamba-ssm from source...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK (native CUDA)')
except ImportError:
    print('mamba-ssm: FAILED -- will use pure-PyTorch SSM fallback')

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

## Configuration

In [ ]:
import os, sys, gc, time
import numpy as np
import pandas as pd

sys.path.insert(0, '.')

OUTPUT_BASE = './results/learned_quantization_bottleneck/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Sequence / discretization ---
SEQ_LEN    = 512
N_BINS     = 256
N_TRAIN    = 50_000
N_EVAL     = 2_000
SEED       = 320

# --- Special tokens ---
PAD_TOKEN  = N_BINS
MASK_TOKEN = N_BINS + 1
VOCAB_SIZE = N_BINS + 2  # 258

# --- Model architecture ---
D_MODEL    = 256
N_LAYERS   = 4
N_HEADS    = 4
FFN_DIM    = 1024
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

# --- Training ---
LR         = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS     = 20
BATCH_SIZE = 64

# --- Perturbation ---
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]

# --- VQ codebook sizes to sweep ---
VQ_CODEBOOK_SIZES = [32, 64, 128, 256, 512, 1024]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
print(f'VQ codebook sizes: {VQ_CODEBOOK_SIZES}')
print(f'Output: {OUTPUT_BASE}')

## Dual-Return Lorenz Generator + VQ Tokenizer

Returns BOTH discrete tokens AND raw continuous values.
Also implements k-means VQ tokenization for Experiment B.

In [ ]:
from scipy.integrate import solve_ivp
from sklearn.cluster import MiniBatchKMeans


def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    """Map continuous values to integer bins. FIX: uses global min/max when provided."""
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, n_bins // 2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.clip((normed * (n_bins - 1)).astype(np.int64), 0, n_bins - 1)


def normalize_to_01(values, global_min=None, global_max=None):
    """Normalize to [0,1]. FIX: uses global min/max when provided."""
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, 0.5, dtype=np.float32)
    return np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0).astype(np.float32)


def _lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]


def generate_lorenz(n_sequences, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    """Returns: (discrete_tokens, continuous_values, raw_floats, global_range).

    raw_floats are un-normalized for VQ fitting.
    uses dataset-global min/max for discretize and normalize.
    """
    rng = np.random.default_rng(seed)
    t_span = (0, 25)
    t_eval = np.linspace(*t_span, seq_len)
    tokens_list, contin_list, raw_list = [], [], []
    for _ in range(n_sequences):
        x0 = rng.uniform(-15, 15)
        y0 = rng.uniform(-15, 15)
        z0 = rng.uniform(10, 40)
        sol = solve_ivp(_lorenz_rhs, t_span, [x0, y0, z0],
                        t_eval=t_eval, method='RK45', max_step=0.05)
        if sol.success and len(sol.y[0]) == seq_len:
            raw = sol.y[0]
        else:
            x0 += rng.uniform(-1, 1)
            sol2 = solve_ivp(_lorenz_rhs, t_span, [x0, y0, z0],
                             t_eval=t_eval, method='RK45', max_step=0.01)
            raw = sol2.y[0][:seq_len]
        raw_list.append(raw.astype(np.float32))

    # Compute global range from all raw values
    all_raw = np.concatenate(raw_list)
    if global_range is None:
        global_range = (float(all_raw.min()), float(all_raw.max()))
    gmin, gmax = global_range

    tokens_list = [discretize(r, global_min=gmin, global_max=gmax) for r in raw_list]
    contin_list = [normalize_to_01(r, global_min=gmin, global_max=gmax) for r in raw_list]

    return (np.array(tokens_list, dtype=np.int64),
            np.array(contin_list, dtype=np.float32),
            np.array(raw_list, dtype=np.float32),
            global_range)


class VQTokenizer:
    """Vector Quantization tokenizer using k-means on 1D values.

    Learns a codebook from the training data, then maps each continuous
    value to its nearest codebook entry. This is the "smart tokenizer"
    that learned quantization advocates would use.
    """
    def __init__(self, n_codes, seed=SEED):
        self.n_codes = n_codes
        self.kmeans = MiniBatchKMeans(
            n_clusters=n_codes, random_state=seed,
            batch_size=4096, n_init=3,
        )
        self.fitted = False

    def fit(self, raw_values):
        """Fit codebook on flat array of continuous values."""
        flat = raw_values.flatten().reshape(-1, 1)
        # Subsample if too large
        if len(flat) > 500_000:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(flat), 500_000, replace=False)
            flat = flat[idx]
        self.kmeans.fit(flat)
        self.fitted = True
        self.codebook = self.kmeans.cluster_centers_.flatten()
        return self

    def encode(self, raw_values):
        """Map raw floats to VQ token indices."""
        shape = raw_values.shape
        flat = raw_values.flatten().reshape(-1, 1)
        tokens = self.kmeans.predict(flat)
        return tokens.reshape(shape).astype(np.int64)

    def decode(self, tokens):
        """Reconstruct continuous values from tokens."""
        return self.codebook[tokens]


# === Generate data ===
print('Generating Lorenz data...')
train_tokens, train_contin, train_raw, LORENZ_RANGE = generate_lorenz(N_TRAIN, seed=SEED)
eval_tokens, eval_contin, eval_raw, _ = generate_lorenz(N_EVAL, seed=SEED + 1000, global_range=LORENZ_RANGE)
print(f'  Global range: ({LORENZ_RANGE[0]:.2f}, {LORENZ_RANGE[1]:.2f})')
print(f'  Train: tokens={train_tokens.shape}, contin={train_contin.shape}')
print(f'  Eval:  tokens={eval_tokens.shape}, contin={eval_contin.shape}')

# === Fit VQ tokenizers at each codebook size ===
vq_tokenizers = {}
for n_codes in VQ_CODEBOOK_SIZES:
    print(f'\n  Fitting VQ tokenizer (K={n_codes})...')
    vq = VQTokenizer(n_codes, seed=SEED)
    vq.fit(train_raw)

    # Encode train and eval
    vq_train = vq.encode(train_raw)
    vq_eval = vq.encode(eval_raw)

    # Reconstruction error
    recon = vq.decode(vq_train)
    mse = np.mean((train_raw - recon) ** 2)
    print(f'    Codebook: {vq.codebook.min():.1f} to {vq.codebook.max():.1f}')
    print(f'    Reconstruction MSE: {mse:.6f}')
    print(f'    Token range: [{vq_train.min()}, {vq_train.max()}]')

    vq_tokenizers[n_codes] = {
        'tokenizer': vq,
        'train_tokens': vq_train,
        'eval_tokens': vq_eval,
        'vocab_size': n_codes + 2,  # +pad +mask
        'recon_mse': mse,
    }

print('\nAll generators and VQ tokenizers ready')

## Perturbation Suite

**Critical design**: For VQ tokens, perturbing in index space is
semantically wrong because adjacent VQ indices are unordered (k-means
labels are arbitrary). Instead, we perturb in CONTINUOUS space
and re-encode through the VQ tokenizer. This ensures perturbation
magnitude is equivalent across all conditions.

Time reversal is excluded from this experiment because it tests global
structural symmetry, not the quantization bottleneck.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: np.ndarray
    params: dict = field(default_factory=dict)
    description: str = ''


def noise_perturb_uniform(sequences, rate, rng, n_bins=N_BINS):
    """Perturb uniform-grid tokens: adjacent bins = adjacent continuous values."""
    out = sequences.copy()
    noise_std = n_bins * 0.10
    mask = rng.random(out.shape) < rate
    noise = rng.normal(0, noise_std, size=out.shape).astype(np.int64)
    out[mask] = np.clip(out[mask] + noise[mask], 0, n_bins - 1)
    return out


def noise_perturb_vq_continuous(raw_values, rate, rng, vq_tokenizer):
    """Perturb in CONTINUOUS space, then re-encode through VQ.

    This ensures the perturbation magnitude is semantically equivalent
    to the uniform-grid perturbation. Perturbing in VQ-index space would
    be meaningless because k-means labels are unordered.
    """
    out = raw_values.copy()
    value_range = raw_values.max() - raw_values.min()
    noise_std = value_range * 0.10
    mask = rng.random(out.shape) < rate
    noise = rng.normal(0, noise_std, size=out.shape).astype(np.float32)
    out[mask] += noise[mask]
    return vq_tokenizer.encode(out)


class UniformPerturbationSuite:
    """Perturbation suite for uniform-grid tokenized sequences."""
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES

    def run_all(self, sequences):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate * 100)}pct'
            results[name] = PerturbedSet(
                name=name, category='noise',
                sequences=noise_perturb_uniform(sequences, rate, self.rng),
                params={'noise_rate': rate},
                description=f'Value noise at {rate*100:.0f}%',
            )
        return results


class VQPerturbationSuite:
    """Perturbation suite for VQ-tokenized sequences.

    Perturbs in continuous space, then re-encodes through the VQ tokenizer.
    This ensures semantically equivalent perturbation magnitude.
    """
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES

    def run_all(self, raw_values, vq_tokenizer):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate * 100)}pct'
            results[name] = PerturbedSet(
                name=name, category='noise',
                sequences=noise_perturb_vq_continuous(
                    raw_values, rate, self.rng, vq_tokenizer),
                params={'noise_rate': rate},
                description=f'Value noise at {rate*100:.0f}% (continuous-domain)',
            )
        return results


print('Perturbation suites ready (uniform + VQ continuous-domain)')

## Model Definitions

Three model variants:
- **SmallBERT** (baseline, categorical CE)
- **SmallBERT_Continuous** (MSE head, scalar output)
- **SmallBERT_VQ** (categorical CE, but with VQ vocab size)

Plus SmallMamba equivalents for comparison.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class SmallBERT(nn.Module):
    """Causal Transformer with categorical head (baseline)."""
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN,
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, x, return_hidden=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        causal_mask = self._causal_mask(L, x.device)
        h = self.encoder(h, mask=causal_mask)
        h = self.norm(h)
        logits = self.head(h)
        if return_hidden:
            return logits, h
        return logits


class SmallBERT_Continuous(nn.Module):
    """SmallBERT with continuous scalar output head (MSE loss).

    Architecture identical to SmallBERT except:
    - head: Linear(d_model, 1) instead of Linear(d_model, vocab_size)
    - Output is a continuous scalar per position
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN,
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, 1)  # KEY: continuous scalar
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _causal_mask(self, seq_len, device):
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, x, return_hidden=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        causal_mask = self._causal_mask(L, x.device)
        h = self.encoder(h, mask=causal_mask)
        h = self.norm(h)
        out = self.head(h).squeeze(-1)  # (B, L) continuous
        if return_hidden:
            return out, h
        return out


# --- SmallMamba variants ---
if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock

    class SmallMamba(nn.Module):
        """4-layer Mamba with categorical head."""
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)
            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(
                    MambaBlock(d_model=d_model, d_state=d_state,
                               d_conv=d_conv, expand=expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            if return_hidden:
                return logits, h
            return logits
else:
    class SimpleSSMLayer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            d_inner = d_model * expand
            self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
            self.conv1d   = nn.Conv1d(d_inner, d_inner, d_conv,
                                      padding=d_conv - 1, groups=d_inner)
            self.x_proj   = nn.Linear(d_inner, d_state * 2, bias=False)
            self.dt_proj  = nn.Linear(d_state, d_inner, bias=True)
            self.A_log    = nn.Parameter(torch.log(torch.randn(d_inner, d_state).abs() + 1e-4))
            self.D        = nn.Parameter(torch.ones(d_inner))
            self.out_proj = nn.Linear(d_inner, d_model, bias=False)

        def forward(self, x):
            B, L, D = x.shape
            xz = self.in_proj(x)
            x_inner, z = xz.chunk(2, dim=-1)
            x_conv = self.conv1d(x_inner.transpose(1, 2))[:, :, :L].transpose(1, 2)
            x_conv = torch.silu(x_conv)
            y = x_conv * torch.silu(z)
            y = y * self.D.unsqueeze(0).unsqueeze(0)
            return self.out_proj(y)

    class SmallMamba(nn.Module):
        """4-layer SSM (PyTorch fallback) with categorical head."""
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)
            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(SimpleSSMLayer(d_model, d_state, d_conv, expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            if return_hidden:
                return logits, h
            return logits
    print('NOTE: Using pure-PyTorch SSM fallback')


# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # No weight tying (vocab_size != d_model)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


class SmallStripedHyena_Continuous(nn.Module):
    """SmallStripedHyena with continuous scalar output head (MSE loss)."""
    def __init__(self, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
                 seq_len=SEQ_LEN, order=2, mlp_ratio=4, attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        # Use token embeddings (same as BERT/Mamba continuous variants)
        self.tok_emb = nn.Embedding(VOCAB_SIZE, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena_Continuous: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.drop(self.tok_emb(x))
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        out = self.head(h).squeeze(-1)
        return (out, h) if return_hidden else out

# Print parameter counts
for name, cls in [('SmallBERT (CE)', SmallBERT),
                  ('SmallBERT_Continuous (MSE)', SmallBERT_Continuous),
                  ('SmallMamba (CE)', SmallMamba),
                  ('SmallStripedHyena (CE)', SmallStripedHyena)]:
    m = cls()
    n_p = sum(p.numel() for p in m.parameters())
    print(f'  {name:35s}: {n_p/1e6:.3f}M params')
    del m

# VQ models have different vocab sizes
for n_codes in VQ_CODEBOOK_SIZES:
    vs = n_codes + 2
    m = SmallBERT(vocab_size=vs)
    n_p = sum(p.numel() for p in m.parameters())
    print(f'  SmallBERT_VQ (K={n_codes:3d}, vocab={vs}): {n_p/1e6:.3f}M params')
    del m

print('\nAll model definitions ready')



## Training Loops (CE + MSE)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def create_causal_batch(x):
    return x[:, :-1], x[:, 1:]


def train_model_ce(model, train_data, val_data, tag, vocab_size=VOCAB_SIZE,
                   epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                   weight_decay=WEIGHT_DECAY, seed=SEED):
    """Train with cross-entropy loss (categorical)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = model.to(DEVICE)
    train_ds = TensorDataset(torch.from_numpy(train_data).long())
    val_ds   = TensorDataset(torch.from_numpy(val_data).long())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader))
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_state = None
    ckpt_path = f'{CKPT_DIR}/{tag}_best.pt'
    if os.path.exists(ckpt_path):
        print(f'  Loading checkpoint: {ckpt_path}')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        return model, [], []

    print(f'  Training {tag} (CE, {epochs} epochs)...')
    start = time.time()
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for (batch_x,) in train_loader:
            batch_x = batch_x.to(DEVICE)
            inputs, targets = create_causal_batch(batch_x)
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            n_batches += 1
        train_losses.append(epoch_loss / n_batches)

        model.eval()
        val_loss, vb = 0.0, 0
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(DEVICE)
                inputs, targets = create_causal_batch(batch_x)
                logits = model(inputs)
                loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
                val_loss += loss.item()
                vb += 1
        avg_val = val_loss / max(vb, 1)
        val_losses.append(avg_val)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'    Epoch {epoch+1:3d}/{epochs}  val={avg_val:.4f}  '
                  f'best={best_val_loss:.4f}  [{time.time()-start:.0f}s]')

    if best_state:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)
    print(f'  Done in {(time.time()-start)/60:.1f}min')
    return model, train_losses, val_losses


def train_model_mse(model, train_tokens, train_contin, val_tokens, val_contin,
                    tag, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                    weight_decay=WEIGHT_DECAY, seed=SEED):
    """Train with MSE loss on raw continuous targets."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = model.to(DEVICE)
    train_ds = TensorDataset(torch.from_numpy(train_tokens).long(),
                             torch.from_numpy(train_contin).float())
    val_ds   = TensorDataset(torch.from_numpy(val_tokens).long(),
                             torch.from_numpy(val_contin).float())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader))
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    ckpt_path = f'{CKPT_DIR}/{tag}_best.pt'
    if os.path.exists(ckpt_path):
        print(f'  Loading checkpoint: {ckpt_path}')
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
        return model, [], []

    print(f'  Training {tag} (MSE on raw continuous, {epochs} epochs)...')
    start = time.time()
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for batch_tok, batch_con in train_loader:
            batch_tok = batch_tok.to(DEVICE)
            batch_con = batch_con.to(DEVICE)
            inputs  = batch_tok[:, :-1]
            targets = batch_con[:, 1:]
            preds = model(inputs)
            loss = criterion(preds, targets)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            n_batches += 1
        train_losses.append(epoch_loss / n_batches)

        model.eval()
        val_loss, vb = 0.0, 0
        with torch.no_grad():
            for batch_tok, batch_con in val_loader:
                batch_tok = batch_tok.to(DEVICE)
                batch_con = batch_con.to(DEVICE)
                inputs  = batch_tok[:, :-1]
                targets = batch_con[:, 1:]
                preds = model(inputs)
                loss = criterion(preds, targets)
                val_loss += loss.item()
                vb += 1
        avg_val = val_loss / max(vb, 1)
        val_losses.append(avg_val)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'    Epoch {epoch+1:3d}/{epochs}  val_mse={avg_val:.6f}  '
                  f'best={best_val_loss:.6f}  [{time.time()-start:.0f}s]')

    if best_state:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)
    print(f'  Done in {(time.time()-start)/60:.1f}min')
    return model, train_losses, val_losses


print('Training loops ready (CE + MSE)')

## Evaluation -- Procrustes + Embedding Extraction

In [ ]:
from scipy.linalg import orthogonal_procrustes


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def extract_embeddings(model, sequences, batch_size=128):
    """Mean-pooled last-layer hidden states."""
    model.eval()
    all_embs = []
    for i in range(0, len(sequences), batch_size):
        batch = torch.from_numpy(sequences[i:i+batch_size]).long().to(DEVICE)
        _, hidden = model(batch, return_hidden=True)
        emb = hidden.mean(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.concatenate(all_embs, axis=0)


def compute_procrustes_distortion(emb_clean, emb_perturbed):
    """Returns (D_normalized, D_unnormalized)."""
    X = emb_clean - emb_clean.mean(axis=0)
    Y = emb_perturbed - emb_perturbed.mean(axis=0)
    X_norm = np.linalg.norm(X, 'fro')
    Y_norm = np.linalg.norm(Y, 'fro')
    if X_norm < 1e-10 or Y_norm < 1e-10:
        return 1.0, float('inf')

    # Unnormalized
    R_raw, _ = orthogonal_procrustes(X, Y)
    D_unnorm = float(np.linalg.norm(X @ R_raw - Y, 'fro'))

    # Normalized
    Xn, Yn = X / X_norm, Y / Y_norm
    R_norm, _ = orthogonal_procrustes(Xn, Yn)
    residual = np.linalg.norm(Xn @ R_norm - Yn, 'fro')
    D_norm = float(np.clip(residual / np.sqrt(2.0), 0, 1))

    return D_norm, D_unnorm


print('Procrustes distortion ready (normalized + unnormalized)')

## Main Experiment

Three conditions:
- **A**: SmallBERT_Continuous (MSE head)
- **B**: SmallBERT with VQ tokens at each codebook size
- **C**: SmallBERT baseline (uniform 256-bin CE)

Plus SmallMamba baseline for reference.

In [ ]:
print('=' * 70)
print('LEARNED QUANTIZATION & CONTINUOUS ABLATIONS')
print('=' * 70)

all_results = []

# ============================================================
# Condition C: Baseline SmallBERT (Uniform 256-bin, CE)
# ============================================================
print('\n--- Condition C: SmallBERT Baseline (Uniform 256-bin, CE) ---')
model_c = SmallBERT()
n_params_c = sum(p.numel() for p in model_c.parameters())
model_c, _, _ = train_model_ce(model_c, train_tokens, eval_tokens,
                               tag='SmallBERT_baseline_lorenz', seed=SEED)

pert_suite = UniformPerturbationSuite(seed=SEED)
perturbed_baseline = pert_suite.run_all(eval_tokens)

emb_clean_c = extract_embeddings(model_c, eval_tokens)
for pert_name, pset in perturbed_baseline.items():
    emb_pert = extract_embeddings(model_c, pset.sequences)
    D_norm, D_unnorm = compute_procrustes_distortion(emb_clean_c, emb_pert)
    all_results.append({
        'condition': 'C_Baseline', 'tokenizer': 'Uniform_256',
        'head': 'CE', 'architecture': 'SmallBERT',
        'n_params': n_params_c, 'codebook_size': 256,
        'perturbation': pert_name,
        'procrustes_D': D_norm, 'procrustes_D_unnorm': D_unnorm,
    })
    print(f'    {pert_name}: D_norm={D_norm:.4f}, D_unnorm={D_unnorm:.2f}')

del model_c
cleanup_gpu()

# ============================================================
# Condition A: SmallBERT_Continuous (MSE head)
# ============================================================
print('\n--- Condition A: SmallBERT + Continuous MSE Head ---')
model_a = SmallBERT_Continuous()
n_params_a = sum(p.numel() for p in model_a.parameters())
model_a, _, _ = train_model_mse(model_a, train_tokens, train_contin,
                                eval_tokens, eval_contin,
                                tag='SmallBERT_continuous_lorenz', seed=SEED)

emb_clean_a = extract_embeddings(model_a, eval_tokens)
for pert_name, pset in perturbed_baseline.items():
    emb_pert = extract_embeddings(model_a, pset.sequences)
    D_norm, D_unnorm = compute_procrustes_distortion(emb_clean_a, emb_pert)
    all_results.append({
        'condition': 'A_Continuous', 'tokenizer': 'Uniform_256',
        'head': 'MSE', 'architecture': 'SmallBERT',
        'n_params': n_params_a, 'codebook_size': 256,
        'perturbation': pert_name,
        'procrustes_D': D_norm, 'procrustes_D_unnorm': D_unnorm,
    })
    print(f'    {pert_name}: D_norm={D_norm:.4f}, D_unnorm={D_unnorm:.2f}')

del model_a
cleanup_gpu()

# ============================================================
# Condition B: SmallBERT with VQ Learned Tokens (CE)
# ============================================================
for n_codes in VQ_CODEBOOK_SIZES:
    vq_info = vq_tokenizers[n_codes]
    vq_vocab = vq_info['vocab_size']
    print(f'\n--- Condition B: SmallBERT + VQ (K={n_codes}) ---')

    model_b = SmallBERT(vocab_size=vq_vocab)
    n_params_b = sum(p.numel() for p in model_b.parameters())
    model_b, _, _ = train_model_ce(
        model_b, vq_info['train_tokens'], vq_info['eval_tokens'],
        tag=f'SmallBERT_VQ{n_codes}_lorenz', vocab_size=vq_vocab, seed=SEED)

    # Perturb in continuous space, re-encode through VQ
    pert_suite_vq = VQPerturbationSuite(seed=SEED)
    perturbed_vq = pert_suite_vq.run_all(eval_raw, vq_info['tokenizer'])

    emb_clean_b = extract_embeddings(model_b, vq_info['eval_tokens'])
    for pert_name, pset in perturbed_vq.items():
        emb_pert = extract_embeddings(model_b, pset.sequences)
        D_norm, D_unnorm = compute_procrustes_distortion(emb_clean_b, emb_pert)
        all_results.append({
            'condition': f'B_VQ_{n_codes}', 'tokenizer': f'VQ_{n_codes}',
            'head': 'CE', 'architecture': 'SmallBERT',
            'n_params': n_params_b, 'codebook_size': n_codes,
            'vq_recon_mse': vq_info['recon_mse'],
            'perturbation': pert_name,
            'procrustes_D': D_norm, 'procrustes_D_unnorm': D_unnorm,
        })
        print(f'    {pert_name}: D_norm={D_norm:.4f}, D_unnorm={D_unnorm:.2f}')

    del model_b
    cleanup_gpu()

# ============================================================
# SmallMamba baseline (for reference)
# ============================================================
print('\n--- SmallMamba Baseline (Uniform 256-bin, CE) ---')
model_m = SmallMamba()
n_params_m = sum(p.numel() for p in model_m.parameters())
model_m, _, _ = train_model_ce(model_m, train_tokens, eval_tokens,
                               tag='SmallMamba_baseline_lorenz', seed=SEED)

emb_clean_m = extract_embeddings(model_m, eval_tokens)
for pert_name, pset in perturbed_baseline.items():
    emb_pert = extract_embeddings(model_m, pset.sequences)
    D_norm, D_unnorm = compute_procrustes_distortion(emb_clean_m, emb_pert)
    all_results.append({
        'condition': 'Mamba_Baseline', 'tokenizer': 'Uniform_256',
        'head': 'CE', 'architecture': 'SmallMamba',
        'n_params': n_params_m, 'codebook_size': 256,
        'perturbation': pert_name,
        'procrustes_D': D_norm, 'procrustes_D_unnorm': D_unnorm,
    })
    print(f'    {pert_name}: D_norm={D_norm:.4f}, D_unnorm={D_unnorm:.2f}')

del model_m
cleanup_gpu()


# ============================================================
# SmallStripedHyena baseline (CE) -- for architecture comparison
# ============================================================
print('\n--- SmallStripedHyena Baseline (Uniform 256-bin, CE) ---')
model_sh = SmallStripedHyena()
n_params_sh = sum(p.numel() for p in model_sh.parameters())
model_sh, _, _ = train_model_ce(model_sh, train_tokens, eval_tokens,
                                tag='SmallStripedHyena_baseline_lorenz', seed=SEED)

emb_clean_sh = extract_embeddings(model_sh, eval_tokens)
for pert_name, pset in perturbed_baseline.items():
    emb_pert = extract_embeddings(model_sh, pset.sequences)
    D_norm, D_unnorm = compute_procrustes_distortion(emb_clean_sh, emb_pert)
    all_results.append({
        'condition': 'StripedHyena_Baseline', 'tokenizer': 'Uniform_256',
        'head': 'CE', 'architecture': 'SmallStripedHyena',
        'n_params': n_params_sh, 'codebook_size': 256,
        'perturbation': pert_name,
        'procrustes_D': D_norm, 'procrustes_D_unnorm': D_unnorm,
    })
    print(f'    {pert_name}: D_norm={D_norm:.4f}, D_unnorm={D_unnorm:.2f}')

del model_sh
cleanup_gpu()

# ============================================================
# SmallStripedHyena + Continuous MSE head
# ============================================================
print('\n--- SmallStripedHyena + Continuous MSE Head ---')
model_sh_cont = SmallStripedHyena_Continuous()
n_params_sh_cont = sum(p.numel() for p in model_sh_cont.parameters())
model_sh_cont, _, _ = train_model_mse(model_sh_cont, train_tokens, train_contin,
                                       eval_tokens, eval_contin,
                                       tag='SmallStripedHyena_continuous_lorenz', seed=SEED)

emb_clean_sh_cont = extract_embeddings(model_sh_cont, eval_tokens)
for pert_name, pset in perturbed_baseline.items():
    emb_pert = extract_embeddings(model_sh_cont, pset.sequences)
    D_norm, D_unnorm = compute_procrustes_distortion(emb_clean_sh_cont, emb_pert)
    all_results.append({
        'condition': 'StripedHyena_Continuous', 'tokenizer': 'Uniform_256',
        'head': 'MSE', 'architecture': 'SmallStripedHyena',
        'n_params': n_params_sh_cont, 'codebook_size': 256,
        'perturbation': pert_name,
        'procrustes_D': D_norm, 'procrustes_D_unnorm': D_unnorm,
    })
    print(f'    {pert_name}: D_norm={D_norm:.4f}, D_unnorm={D_unnorm:.2f}')

del model_sh_cont
cleanup_gpu()

# Save results
df_results = pd.DataFrame(all_results)
results_path = f'{RESULTS_DIR}/learned_quantization_bottleneck.csv'
df_results.to_csv(results_path, index=False)
print(f'\nSaved: {results_path}')
print(f'\n{df_results.to_string(index=False)}')

## The Bottleneck -- Visualization

Three panels:
- **A**: Procrustes D comparison across conditions (bar chart)
- **B**: VQ codebook scaling — D vs codebook size (shows delay, not elimination)
- **C**: Phase portrait comparison (Poincaré cross-sections)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

# --- Aggregate ---
df_agg = df_results.groupby(['condition', 'tokenizer', 'head']).agg(
    mean_D=('procrustes_D', 'mean'),
    std_D=('procrustes_D', 'std'),
    mean_D_unnorm=('procrustes_D_unnorm', 'mean'),
).reset_index()

# ===== Figure 1: The Bottleneck (3-panel) =====
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# --- Panel A: Bar chart comparing mean D across conditions ---
ax = axes[0]
conditions_order = ['A_Continuous', 'C_Baseline', 'Mamba_Baseline',
                    'StripedHyena_Baseline', 'StripedHyena_Continuous']
conditions_order += [f'B_VQ_{k}' for k in VQ_CODEBOOK_SIZES]

colors = {
    'A_Continuous': '#16A34A',    # green (best)
    'C_Baseline': '#DC2626',      # red (baseline)
    'Mamba_Baseline': '#7C3AED',  # purple
    'StripedHyena_Baseline': '#0EA5E9',  # sky blue
    'StripedHyena_Continuous': '#06B6D4',  # cyan
}
for k in VQ_CODEBOOK_SIZES:
    colors[f'B_VQ_{k}'] = '#F59E0B'  # amber (VQ variants)

x_pos = []
bar_colors = []
bar_labels = []
bar_vals = []
bar_errs = []

for i, cond in enumerate(conditions_order):
    row = df_agg[df_agg['condition'] == cond]
    if row.empty:
        continue
    x_pos.append(i)
    bar_colors.append(colors.get(cond, '#94A3B8'))
    bar_labels.append(cond.replace('_', '\n'))
    bar_vals.append(row['mean_D'].values[0])
    bar_errs.append(row['std_D'].values[0])

ax.bar(x_pos, bar_vals, yerr=bar_errs, color=bar_colors, edgecolor='black',
       linewidth=0.8, capsize=5, alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(bar_labels, fontsize=8, rotation=30, ha='right')
ax.set_ylabel('Mean Procrustes Distortion $D$', fontsize=12)
ax.set_title('A. The Bottleneck Proof\n'
             'Continuous MSE eliminates fracture; VQ delays it',
             fontweight='bold', fontsize=13)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.15, axis='y')

# --- Panel B: VQ codebook scaling + Shannon curve fit ---
ax = axes[1]
vq_rows = df_agg[df_agg['condition'].str.startswith('B_VQ_')].copy()
if not vq_rows.empty:
    vq_rows['codebook_size'] = vq_rows['condition'].str.extract(r'VQ_(\d+)').astype(int)
    vq_rows = vq_rows.sort_values('codebook_size')

    ax.errorbar(vq_rows['codebook_size'], vq_rows['mean_D'],
                yerr=vq_rows['std_D'], fmt='o-', color='#F59E0B',
                linewidth=2.5, markersize=10, capsize=6, capthick=2,
                label='VQ Learned (CE)')

    # Fit log vs polynomial to prove Shannon bound
    K_vals = vq_rows['codebook_size'].values.astype(float)
    D_vals = vq_rows['mean_D'].values
    if len(K_vals) >= 3:
        # Log fit: D ~ a / log(K) + b (Shannon prediction)
        log_K = np.log(K_vals)
        slope_log, intercept_log, r_log, _, _ = stats.linregress(1.0 / log_K, D_vals)
        # Polynomial fit: D ~ a / K + b (naive expectation)
        slope_poly, intercept_poly, r_poly, _, _ = stats.linregress(1.0 / K_vals, D_vals)

        K_smooth = np.linspace(K_vals.min() * 0.8, K_vals.max() * 1.5, 100)
        D_log_fit = slope_log / np.log(K_smooth) + intercept_log
        D_poly_fit = slope_poly / K_smooth + intercept_poly

        ax.plot(K_smooth, D_log_fit, '--', color='#DC2626', linewidth=2,
                alpha=0.7, label=f'$D \\sim 1/\\log(K)$ (R²={r_log**2:.3f})')
        ax.plot(K_smooth, D_poly_fit, ':', color='#7C3AED', linewidth=2,
                alpha=0.5, label=f'$D \\sim 1/K$ (R²={r_poly**2:.3f})')

    # Add baseline and continuous reference lines
    baseline_D = df_agg[df_agg['condition'] == 'C_Baseline']['mean_D'].values
    contin_D = df_agg[df_agg['condition'] == 'A_Continuous']['mean_D'].values
    if len(baseline_D) > 0:
        ax.axhline(y=baseline_D[0], color='#DC2626', linewidth=1.5, linestyle='--',
                   alpha=0.4, label=f'Baseline Uniform (D={baseline_D[0]:.3f})')
    if len(contin_D) > 0:
        ax.axhline(y=contin_D[0], color='#16A34A', linewidth=1.5, linestyle='--',
                   alpha=0.4, label=f'Continuous MSE (D={contin_D[0]:.3f})')

ax.set_xlabel('VQ Codebook Size $K$', fontsize=12)
ax.set_ylabel('Mean Procrustes Distortion $D$', fontsize=12)
ax.set_title('B. Shannon Bound: $D \\sim 1/\\log(K)$\n'
             'Logarithmic, not polynomial, decay',
             fontweight='bold', fontsize=13)
ax.set_xscale('log', base=2)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.15)

# --- Panel C: Unnormalized D (catches scale drift) ---
ax = axes[2]
for cond in conditions_order:
    row = df_agg[df_agg['condition'] == cond]
    if row.empty:
        continue
    ax.bar(cond.replace('_', '\n'), row['mean_D_unnorm'].values[0],
           color=colors.get(cond, '#94A3B8'), edgecolor='black',
           linewidth=0.8, alpha=0.85)

ax.set_ylabel('Unnormalized Procrustes $D_{raw}$', fontsize=12)
ax.set_title('C. Scale Drift (Unnormalized)\n'
             'Catches ESM-2-style scale collapse',
             fontweight='bold', fontsize=13)
ax.tick_params(axis='x', rotation=30, labelsize=8)
ax.grid(True, alpha=0.15, axis='y')

fig.suptitle('The Discrete Bottleneck Proof:\n'
             'Continuous Heads Eliminate Fracture, Learned Tokens Only Delay It',
             fontsize=14, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/bottleneck_proof.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.savefig(f'{RESULTS_DIR}/bottleneck_proof.pdf', bbox_inches='tight')
plt.show()
print(f'\nSaved: {RESULTS_DIR}/bottleneck_proof.png')

## Embedding-Space Visualization

**Why not autoregressive Poincaré?** For the continuous MSE model,
autoregressive generation requires re-discretizing the scalar output
back to a token for the next input step. This re-introduces the
discretization bottleneck at inference time, undermining the whole
proof. Instead, we show **embedding-space PCA** of clean vs perturbed
sequences, which directly visualizes manifold stability without
re-discretization artifacts.

For the categorical models (baseline + VQ), we DO show autoregressive
Poincaré because discretization is inherent to their design.

In [ ]:
from sklearn.decomposition import PCA

@torch.no_grad()
def generate_sequence_categorical(model, seed_tokens, gen_length, vocab_size=N_BINS):
    """Autoregressively generate from a categorical model."""
    model.eval()
    tokens = torch.from_numpy(seed_tokens[:50]).long().unsqueeze(0).to(DEVICE)
    generated = list(seed_tokens[:50])
    for _ in range(gen_length):
        logits = model(tokens)[0, -1, :]
        probs = torch.softmax(logits / 0.8, dim=-1)
        pred_token = torch.multinomial(probs, 1).item()
        pred_token = min(pred_token, vocab_size - 1)
        generated.append(pred_token)
        tokens = torch.cat([tokens, torch.tensor([[pred_token]], device=DEVICE)], dim=1)
        if tokens.shape[1] > SEQ_LEN:
            tokens = tokens[:, -SEQ_LEN:]
    return np.array(generated)


print('Generating visualizations...')

# Reload trained models
model_baseline = SmallBERT()
ckpt_base = f'{CKPT_DIR}/SmallBERT_baseline_lorenz_best.pt'
if os.path.exists(ckpt_base):
    model_baseline.load_state_dict(torch.load(ckpt_base, map_location=DEVICE, weights_only=True))
model_baseline = model_baseline.to(DEVICE)

model_continuous = SmallBERT_Continuous()
ckpt_cont = f'{CKPT_DIR}/SmallBERT_continuous_lorenz_best.pt'
if os.path.exists(ckpt_cont):
    model_continuous.load_state_dict(torch.load(ckpt_cont, map_location=DEVICE, weights_only=True))
model_continuous = model_continuous.to(DEVICE)

best_vq_k = VQ_CODEBOOK_SIZES[-1]
vq_vocab = best_vq_k + 2
model_vq = SmallBERT(vocab_size=vq_vocab)
ckpt_vq = f'{CKPT_DIR}/SmallBERT_VQ{best_vq_k}_lorenz_best.pt'
if os.path.exists(ckpt_vq):
    model_vq.load_state_dict(torch.load(ckpt_vq, map_location=DEVICE, weights_only=True))
model_vq = model_vq.to(DEVICE)

# Load StripedHyena models for visualization
model_sh_base = SmallStripedHyena()
ckpt_sh = f'{CKPT_DIR}/SmallStripedHyena_baseline_lorenz_best.pt'
if os.path.exists(ckpt_sh):
    model_sh_base.load_state_dict(torch.load(ckpt_sh, map_location=DEVICE, weights_only=True))
model_sh_base = model_sh_base.to(DEVICE)

model_sh_cont = SmallStripedHyena_Continuous()
ckpt_sh_c = f'{CKPT_DIR}/SmallStripedHyena_continuous_lorenz_best.pt'
if os.path.exists(ckpt_sh_c):
    model_sh_cont.load_state_dict(torch.load(ckpt_sh_c, map_location=DEVICE, weights_only=True))
model_sh_cont = model_sh_cont.to(DEVICE)

# === Figure: 2-row layout ===
# Row 1: Embedding PCA (clean vs 5% noise) for 3 model conditions
# Row 2: Autoregressive Poincaré for categorical models + MSE explanation
fig, axes = plt.subplots(2, 5, figsize=(30, 12))

# --- Row 1: Embedding-space PCA ---
# Note: We drop the "ground truth" token-space PCA because it measures
# sequence-space distances (dim=512), which is fundamentally different
# from embedding-space PCA (dim=256). Comparing them is misleading.
pert_5pct = perturbed_baseline['noise_5pct'].sequences
models_for_pca = [
    ('SmallBERT Baseline\n(CE)', model_baseline, eval_tokens, pert_5pct),
    ('SmallBERT + MSE\n(Continuous)', model_continuous, eval_tokens, pert_5pct),
    (f'SmallBERT + VQ\n(K={best_vq_k})', model_vq,
     vq_tokenizers[best_vq_k]['eval_tokens'],
     # Use same seed as experiment for consistency
     VQPerturbationSuite(seed=SEED).run_all(
         eval_raw, vq_tokenizers[best_vq_k]['tokenizer'])['noise_5pct'].sequences),
    ('StripedHyena Baseline\n(CE)', model_sh_base, eval_tokens, pert_5pct),
    ('StripedHyena + MSE\n(Continuous)', model_sh_cont, eval_tokens, pert_5pct),
]

for col, (title, model, clean_seq, pert_seq) in enumerate(models_for_pca):
    ax = axes[0, col]
    n_show = min(500, len(clean_seq))
    emb_c = extract_embeddings(model, clean_seq[:n_show])
    emb_p = extract_embeddings(model, pert_seq[:n_show])
    pca = PCA(n_components=2)
    clean_2d = pca.fit_transform(emb_c)
    pert_2d = pca.transform(emb_p)

    ax.scatter(clean_2d[:, 0], clean_2d[:, 1], c='#2563EB', alpha=0.3,
              s=10, label='Clean')
    ax.scatter(pert_2d[:, 0], pert_2d[:, 1], c='#DC2626', alpha=0.3,
              s=10, label='5% noise')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(fontsize=8, markerscale=2)
    ax.grid(True, alpha=0.1)

axes[0, 0].text(-0.15, 0.5, 'Embedding PCA\n(clean vs perturbed)',
                transform=axes[0, 0].transAxes, fontsize=12, fontweight='bold',
                va='center', ha='right', rotation=90)

# --- Row 2: Autoregressive Poincaré for categorical models ---
seed_seq = eval_tokens[0]
gen_len = 2000
seq_true = eval_raw[0]

def tokens_to_continuous(tokens, vmin=-20, vmax=20):
    return vmin + (tokens / (N_BINS - 1)) * (vmax - vmin)

seq_baseline_gen = generate_sequence_categorical(model_baseline, seed_seq, gen_len)
seq_vq_gen = generate_sequence_categorical(
    model_vq, vq_tokenizers[best_vq_k]['eval_tokens'][0], gen_len, vocab_size=best_vq_k)

seq_sh_gen = generate_sequence_categorical(model_sh_base, seed_seq, gen_len)

poincare_data = [
    ('Ground Truth\n(ODE)', seq_true[:gen_len], '#0F172A'),
    ('Baseline CE\n(autoregressive)', tokens_to_continuous(seq_baseline_gen[50:]), '#DC2626'),
    (f'VQ K={best_vq_k}\n(autoregressive)', tokens_to_continuous(seq_vq_gen[50:]), '#F59E0B'),
    ('StripedHyena CE\n(autoregressive)', tokens_to_continuous(seq_sh_gen[50:]), '#0EA5E9'),
]

for col, (title, data, color) in enumerate(poincare_data):
    ax = axes[1, col]
    N = len(data)
    if N > 2:
        ax.plot(data[:N-1], data[1:N], '.', color=color, alpha=0.15, markersize=1)
    ax.set_xlabel('$x(t)$')
    ax.set_ylabel('$x(t+1)$')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.grid(True, alpha=0.1)

# Remove duplicate -- keep only the one after Row 2
axes[1, 0].text(-0.15, 0.5, 'Poincaré Section\n(autoregressive)',
                transform=axes[1, 0].transAxes, fontsize=12, fontweight='bold',
                va='center', ha='right', rotation=90)

fig.suptitle('The Discrete Bottleneck Proof: Embedding Stability + Phase Portraits\n'
             'Row 1: Embedding PCA (all 5 conditions)  |  '
             'Row 2: Poincaré (categorical only — MSE omitted: '
             'autoregressive generation re-discretizes, defeating the proof)',
             fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/bottleneck_visual_proof.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.savefig(f'{RESULTS_DIR}/bottleneck_visual_proof.pdf', bbox_inches='tight')
plt.show()
print(f'\nSaved: {RESULTS_DIR}/bottleneck_visual_proof.png')

# Hide unused subplot in row 2
axes[1, 4].set_visible(False)

del model_baseline, model_continuous, model_vq, model_sh_base, model_sh_cont
cleanup_gpu()

---

## Interpretation

If Condition A (MSE) shows dramatically lower distortion than C (baseline),
we have proven that:

1. The **attention mechanism can faithfully represent continuous geometry** —
   the Transformer backbone preserves manifold structure when freed from
   the categorical output constraint.

2. The **bottleneck is at output discretization**: specifically, the
   cross-entropy loss forcing the model to commit to discrete bins.
   Input discretization (token embeddings) and internal processing
   (self-attention) are NOT the cause.

3. **VQ codebooks** improve encoding efficiency ($B_{arch}$) but cannot
   escape the fundamental logarithmic bound:

$$\Phi \approx B_{arch} \cdot \log(S) - \tau_{geom}$$

The takeaway: "While learned quantization (VQ) improves the encoding
efficiency scalar, the architecture remains inextricably trapped by
the logarithmic capacity bound dictated by Shannon's theorem."

## Perturbation asymmetry note

Conditions A and C use uniform-grid token noise (adjacent bins = adjacent
continuous values). Condition B uses continuous-domain noise re-encoded
through VQ. These have subtly different effective magnitudes at the model
input. This is by design: VQ index-space noise is semantically meaningless
(k-means labels are unordered), so continuous-domain perturbation is the
only valid option. The comparison is not perfectly apples-to-apples in
perturbation magnitude, but the alternative (VQ index noise) would be
worse — it would test random reassignment rather than local perturbation.